# 2/2 - Benchmark a checkpoint

Scores a trained checkpoint on the full PU-Net grid and prints the row into
the published comparison table.

## Before running
1. *Add Input* -> `pointdenoise-code`
2. *Add Input* -> `pointdenoise-data`
3. *Add Input* -> the dataset holding your `best.pt`
4. GPU on

Roughly an hour. Calibration runs first and asserts, because a harness that
does not reproduce a published number produces results comparable to nothing.


In [ ]:
import glob, os, subprocess, sys

def find_dir(marker, root="/kaggle/input"):
    for base, dirs, files in os.walk(root):
        if marker in dirs or marker in files:
            return base
    return None

CODE = find_dir("pointdenoise")
DATA = find_dir("examples")
print("code:", CODE)
print("data:", DATA)
assert CODE, f"pointdenoise package not found. /kaggle/input holds: {os.listdir('/kaggle/input')}"
assert DATA, "benchmark data not found (looking for an 'examples' directory)"

sys.path.insert(0, CODE)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "trimesh", "rtree"], check=False)

import torch
print("\ntorch", torch.__version__, "| CUDA", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU - turn it on in Settings")


In [ ]:
CKPT = None
for base, _, files in os.walk("/kaggle/input"):
    if "best.pt" in files:
        CKPT = os.path.join(base, "best.pt"); break
assert CKPT, "best.pt not found - add the dataset holding your checkpoint"
print("checkpoint:", CKPT)

from pointdenoise.engine import load_model
model, ck = load_model(CKPT)
print(f"epoch {ck.get('epoch')}, best loss {ck.get('best'):.6f}, kwargs {ck.get('model_kwargs')}")


## Calibrate first

Scores the bilateral filter, whose numbers are in the published table. Each
metric is checked separately: run 1 passed on CD at 0.84x while P2M sat at
0.17x, so P2M measures something different from what the papers report and
must not be quoted until that is resolved.


In [ ]:
from pointdenoise.benchmark import calibrate, load_released_set

case = load_released_set(DATA, "PUNet", "sparse", 0.01)
r = calibrate(case)
for m in ("cd", "p2m"):
    print(f"  {m.upper():<4} ours {r['measured_'+m]:7.3f}  published {r['expected_'+m]:6.2f}"
          f"  ratio {r[m+'_ratio']:.2f}x  {'PASS' if r[m+'_ok'] else 'FAIL'}")
print(f"\n  quotable: {[m.upper() for m in r['comparable_metrics']]}")
print(f"  not quotable: {[m.upper() for m in r['uncalibrated_metrics']]}")
assert r["cd_ok"], "CD calibration failed - results would not be comparable"


In [ ]:
import numpy as np
from pointdenoise.benchmark import NOISE_LEVELS, comparison_table, run_case
from pointdenoise.data import Shape
from pointdenoise.engine import denoise_cloud

def denoiser(points):
    shape = Shape(np.asarray(points), noisy=np.asarray(points))
    # batch_size drives a B x N x N x 64 tensor inside the attention bias:
    # at N=256 that is 34 MB per sample, so 512 would need 8.6 GB and OOMs.
    return denoise_cloud(model, shape, points_per_patch=256, batch_size=128, iters=1)

scores, baseline = {}, {}
for resolution in ("sparse", "dense"):
    for noise in NOISE_LEVELS:
        try:
            case = load_released_set(DATA, "PUNet", resolution, noise)
        except FileNotFoundError as e:
            print("skip", resolution, noise, e); continue
        _, ours = run_case(case, denoiser, with_p2m=True)
        _, none = run_case(case, lambda p: p, with_p2m=True)
        scores[(resolution, noise)] = ours
        baseline[(resolution, noise)] = none
        gain = (none['cd'] - ours['cd']) / none['cd'] * 100
        print(f"{case.label:<20} ours CD {ours['cd']:7.4f}  noisy CD {none['cd']:7.4f}  {gain:+5.1f}%")


In [ ]:
table = comparison_table(scores, our_name="Ours", dataset="PUNet")
print(table)
print("\nCD only - P2M is not calibrated against the published definition.\n")
print(f"{'case':<20}{'ours CD':>10}{'noisy CD':>10}{'gain':>8}")
for k in scores:
    o, n = scores[k]['cd'], baseline[k]['cd']
    print(f"{k[0]+'/'+format(k[1],'.0%'):<20}{o:>10.4f}{n:>10.4f}{(n-o)/n*100:>7.1f}%")

with open("/kaggle/working/benchmark.txt", "w") as f:
    f.write(table + "\n\nNoisy input baseline (CD)\n")
    for k, v in baseline.items():
        f.write(f"  {k[0]}/{k[1]:.0%}  CD {v['cd']:.4f}\n")
    f.write("\nP2M omitted: not calibrated against the published definition.\n")
print("\nsaved /kaggle/working/benchmark.txt")
